# Actividad 4 - Comparación de modelos preentrenados en Hugging Face

**Rama seleccionada:** procesamiento de lenguaje natural (NLP).  
**Caso de uso:** clasificación de sentimiento en comentarios de clientes de retail omnicanal.  

Este notebook implementa tres modelos preentrenados de Hugging Face, calcula métricas de desempeño, mide latencia y genera tablas/resúmenes para el reporte técnico.

In [ ]:
# 1. Configuración de GPU e instalación de dependencias
!pip install -q transformers datasets evaluate accelerate scikit-learn pandas matplotlib seaborn huggingface_hub

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import pipeline

print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = 0 if torch.cuda.is_available() else -1

## Autenticación en Hugging Face Hub

La mayoría de los modelos usados son públicos. La autenticación es opcional, pero se deja la celda para cumplir el flujo requerido del ecosistema Hugging Face y evitar límites de descarga.

In [ ]:
# 2. Autenticación opcional en Hugging Face Hub
# En Colab: guarda tu token como HF_TOKEN en Secrets, o pega el token manualmente si tu profesor lo solicita.
try:
    from google.colab import userdata
    from huggingface_hub import login
    token = userdata.get("HF_TOKEN")
    if token:
        login(token=token)
        print("Autenticación realizada con HF_TOKEN.")
    else:
        print("Sin token. Se usarán modelos públicos sin autenticación.")
except Exception as e:
    print("Autenticación omitida:", e)

## Dataset de soporte

El dataset es sintético académico y contiene comentarios de clientes de retail, con etiquetas binarias: `positive` y `negative`. La finalidad es contar con un conjunto reproducible, pequeño y controlado para comparar modelos preentrenados.

In [ ]:
# 3. Dataset reproducible
registros = [
    ("T001","The delivery arrived earlier than promised and the driver was very polite.","positive","customer_review"),
    ("T002","The app was easy to use and the checkout process was fast.","positive","customer_review"),
    ("T003","My order was complete and the products were in perfect condition.","positive","customer_review"),
    ("T004","Customer support solved my issue quickly and clearly.","positive","support_ticket"),
    ("T005","The tracking updates were accurate and helpful.","positive","customer_review"),
    ("T006","I liked the discounts and the delivery window was respected.","positive","customer_review"),
    ("T007","The replacement process was simple and the agent was helpful.","positive","support_ticket"),
    ("T008","The website recommended products that were useful for my purchase.","positive","customer_review"),
    ("T009","The package arrived safely and everything matched the description.","positive","customer_review"),
    ("T010","The store pickup experience was quick and well organized.","positive","customer_review"),
    ("T011","The service was reliable and the order status was clear.","positive","customer_review"),
    ("T012","I received a friendly response and the problem was fixed.","positive","support_ticket"),
    ("T013","The product quality was better than expected.","positive","customer_review"),
    ("T014","The delivery notifications helped me plan my day.","positive","customer_review"),
    ("T015","The refund was processed fast and without complications.","positive","support_ticket"),
    ("T016","The item arrived late and customer service did not help.","negative","customer_review"),
    ("T017","The app crashed several times during checkout.","negative","customer_review"),
    ("T018","My order was missing products and nobody explained why.","negative","customer_review"),
    ("T019","The delivery window was missed and the tracking information was wrong.","negative","customer_review"),
    ("T020","The support agent gave confusing instructions.","negative","support_ticket"),
    ("T021","The package was damaged and the return process was slow.","negative","customer_review"),
    ("T022","The website showed inventory available but the product was out of stock.","negative","customer_review"),
    ("T023","The driver could not find the address and the order was cancelled.","negative","customer_review"),
    ("T024","I waited too long for a response from customer support.","negative","support_ticket"),
    ("T025","The recommendation system suggested irrelevant items.","negative","customer_review"),
    ("T026","The pickup location was disorganized and staff could not find the order.","negative","customer_review"),
    ("T027","The refund took too long and I had to contact support twice.","negative","support_ticket"),
    ("T028","The product description was misleading and the item did not meet expectations.","negative","customer_review"),
    ("T029","The checkout failed and I lost the promotion.","negative","customer_review"),
    ("T030","The delivery fee changed unexpectedly at the last step.","negative","customer_review"),
]

df = pd.DataFrame(registros, columns=["id", "text", "label", "source"])
df.to_csv("customer_sentiment_dataset.csv", index=False)
df.head()

In [ ]:
# 4. Modelos seleccionados
modelos = [
    {
        "nombre": "DistilBERT SST-2",
        "hf_id": "distilbert/distilbert-base-uncased-finetuned-sst-2-english",
        "tamano_estimado": "~67M parámetros",
        "licencia": "Apache-2.0",
        "criterio": "Modelo ligero, rápido y adecuado para MVP."
    },
    {
        "nombre": "BERT SST-2",
        "hf_id": "textattack/bert-base-uncased-SST-2",
        "tamano_estimado": "~110M parámetros",
        "licencia": "Revisar model card antes de producción",
        "criterio": "Modelo base más grande; útil para comparar precisión vs. latencia."
    },
    {
        "nombre": "Twitter RoBERTa Sentiment",
        "hf_id": "cardiffnlp/twitter-roberta-base-sentiment-latest",
        "tamano_estimado": "~125M parámetros",
        "licencia": "CC-BY-4.0",
        "criterio": "Entrenado para texto social/informal; útil para comentarios cortos de clientes."
    }
]

pd.DataFrame(modelos)

In [ ]:
# 5. Funciones de inferencia, normalización y evaluación

def normalizar_etiqueta(label):
    label = str(label).lower()
    if "positive" in label or label in ["pos", "label_1"]:
        return "positive"
    if "negative" in label or label in ["neg", "label_0"]:
        return "negative"
    if "neutral" in label:
        return "neutral"
    return label


def evaluar_modelo(modelo_info, textos, y_real, batch_size=8):
    print("Evaluando:", modelo_info["hf_id"])
    clasificador = pipeline(
        "text-classification",
        model=modelo_info["hf_id"],
        device=device,
        truncation=True,
        batch_size=batch_size
    )

    # Warm-up para estabilizar medición de latencia
    _ = clasificador([textos[0]])

    inicio = time.perf_counter()
    salidas = clasificador(textos)
    fin = time.perf_counter()

    y_pred = [normalizar_etiqueta(s["label"]) for s in salidas]
    latencia_ms = ((fin - inicio) / len(textos)) * 1000

    etiquetas = ["negative", "positive"]
    acc = accuracy_score(y_real, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_real, y_pred,
        labels=etiquetas,
        average="macro",
        zero_division=0
    )

    return {
        "modelo": modelo_info["nombre"],
        "hf_id": modelo_info["hf_id"],
        "tamano_estimado": modelo_info["tamano_estimado"],
        "licencia": modelo_info["licencia"],
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
        "latencia_ms_por_texto": latencia_ms,
        "predicciones": y_pred,
        "salidas_raw": salidas
    }

textos = df["text"].tolist()
y_real = df["label"].tolist()

In [ ]:
# 6. Ejecución de modelos
resultados = []

def limpiar_resultado(r):
    return {k:v for k,v in r.items() if k not in ["predicciones", "salidas_raw"]}

for modelo in modelos:
    resultado = evaluar_modelo(modelo, textos, y_real)
    resultados.append(resultado)

df_resultados = pd.DataFrame([limpiar_resultado(r) for r in resultados])
df_resultados

In [ ]:
# 7. Guardar tablas de resultados y predicciones
os.makedirs("results", exist_ok=True)
df_resultados.to_csv("results/resultados_modelos.csv", index=False)

for r in resultados:
    df_pred = df.copy()
    df_pred["prediccion"] = r["predicciones"]
    df_pred["acierto"] = df_pred["label"] == df_pred["prediccion"]
    nombre_archivo = r["modelo"].lower().replace(" ", "_").replace("/", "_")
    df_pred.to_csv(f"results/predicciones_{nombre_archivo}.csv", index=False)

print("Archivos generados en /content/results")

In [ ]:
# 8. Visualización de resultados
plt.figure(figsize=(9, 4))
plt.bar(df_resultados["modelo"], df_resultados["f1_macro"])
plt.title("Comparación de F1-score macro")
plt.ylabel("F1-score macro")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
plt.scatter(df_resultados["latencia_ms_por_texto"], df_resultados["f1_macro"], s=160)
for _, fila in df_resultados.iterrows():
    plt.annotate(fila["modelo"], (fila["latencia_ms_por_texto"], fila["f1_macro"]), textcoords="offset points", xytext=(5,5))
plt.title("Trade-off: latencia vs. F1-score")
plt.xlabel("Latencia promedio por texto (ms)")
plt.ylabel("F1-score macro")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Conclusiones parciales

- El modelo recomendado debe balancear precisión, latencia, licencia y facilidad de despliegue.
- Si la prioridad es eficiencia, DistilBERT suele ser el mejor punto de partida.
- Si la prioridad es precisión y hay GPU disponible, BERT o RoBERTa pueden aportar mayor desempeño, aunque con mayor costo computacional.
- Para producción, se recomienda validar con datos reales, revisar licencias, medir sesgos y monitorear deriva de lenguaje.